# OMS SDK 全量演示 — 纯 MC / MaxFrame AI Function

与 DataWorks **`sdk_*_node.py`** 同路径：`RunContext` + atomic capabilities。

| 步骤 | Capability | 产物 |
|------|------------|------|
| 1 | `extract_clips` | `audio.wav`、Mel PNG/矩阵、MP4、`clips_index.jsonl` |
| 2 | `transcribe_clips` | `asr.jsonl`（MC modelset ASR） |
| 3 | `label_clips` | `labels.jsonl`（MC Omni / fallback VL） |
| 4 | `embed_clips` | `fusion_embeddings.jsonl`（MC VL embedding） |
| 5 | `materialize_preview` | `preview/` 目录 |

> **仅 `MODEL_BACKEND=mc`**。需 `ODPS_*` + `pip install -e "../../piplinesdk[mc]"`（Python 3.11/3.12）。
>
> CLI 等价：`py -3 run_pipeline.py`

## 0. 环境（强制 MC）

复制 `.env.example` → `.env`，或复用仓库根 `.env` 的 `ODPS_*` / `OSS_*`。

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent.parent if (NOTEBOOK_DIR.parent.parent / "piplinesdk").is_dir() else NOTEBOOK_DIR
SDK_ROOT = REPO_ROOT / "piplinesdk"
OUT_DIR = NOTEBOOK_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in (str(SDK_ROOT), str(NOTEBOOK_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv

for env_path in (REPO_ROOT / ".env", NOTEBOOK_DIR / ".env"):
    if env_path.is_file():
        load_dotenv(env_path, override=False)
        print(f"loaded: {env_path}")

from sdk_node_common import load_local_env, resolve_backend, validate_mc_backend

load_local_env()
os.environ["MODEL_BACKEND"] = "mc"
assert resolve_backend() == "mc", "本 notebook 仅支持 MODEL_BACKEND=mc"
validate_mc_backend()

print("MODEL_BACKEND = mc")
print("ODPS_PROJECT  =", os.getenv("ODPS_PROJECT"))
print("ODPS_ACCESS   =", "OK" if os.getenv("ODPS_ACCESS_ID") else "MISSING")
print("OSS_BUCKET    =", os.getenv("OSS_BUCKET"))
print("MC_MODELSET   =", os.getenv("MC_MODELSET_PROJECT", "bigdata_public_modelset"))

In [ ]:
BAG_PATH = Path(os.getenv("BAG_LOCAL_PATH", os.getenv("SDK_DEMO_BAG", r"D:\cursor_project\rosbag_to_labels_pipline\rosbag\output.bag"))).expanduser().resolve()
RUN_OUT = Path(os.getenv("RUN_OUT_DIR", str(OUT_DIR / "run_demo"))).resolve()
CLIP_ID = os.getenv("CLIP_ID", "sha256:notebook-demo")
RUN_ID = os.getenv("RUN_ID", "notebook-demo-001")
MAX_CLIPS = int(os.getenv("SDK_DEMO_MAX_CLIPS", "1"))
RUN_OUT.mkdir(parents=True, exist_ok=True)

print("BAG_PATH =", BAG_PATH)
print("RUN_OUT  =", RUN_OUT)
print("CLIP_ID  =", CLIP_ID)
print("RUN_ID   =", RUN_ID)
assert BAG_PATH.is_file(), f"bag not found: {BAG_PATH}"

## 1. MC Client + RunContext

与 `sdk_node_common.build_sdk_client()` / `make_run_context()` 一致；`media_mode=local` 表示 extract 产物在 `_sdk_work`，MC AI 读本地路径或 base64。

In [ ]:
from oms_multimodal import (
    AcousticPanelConfig,
    AsrConfig,
    ClipConfig,
    ClipVideoConfig,
    __version__,
    bundled_taxonomy_path,
    inspect_bag,
)
from sdk_node_common import build_sdk_client, make_run_context

acoustic_cfg = AcousticPanelConfig(panel_type="mel", n_mels=128, export_mel_matrix=True)
video_cfg = ClipVideoConfig.from_env()
video_cfg.enabled = True
asr_cfg = AsrConfig.from_env()
asr_cfg.enabled = True

client, _, mc_cfg = build_sdk_client(backend="mc", require_taxonomy=True, work_dir=RUN_OUT / "_sdk_work")
client.acoustic_panel_config = acoustic_cfg
client.clip_video_config = video_cfg
client.asr_config = asr_cfg

clip_cfg = ClipConfig(min_sec=15.0, max_sec=20.0, sample_fps=1.0, max_clips=MAX_CLIPS)
ctx = make_run_context(client, RUN_OUT, clip_id=CLIP_ID, run_id=RUN_ID, media_mode="local")

print("SDK", __version__, "backend=", client.model_backend)
print("modelset:", mc_cfg.modelset_project if mc_cfg else "-")
print("run_dir:", ctx.run_dir)
print("clips_index:", ctx.clips_index_path)

## 2. sdk_extract — 中间产物（无 AI）

In [ ]:
from oms_multimodal import extract_clips
from oms_multimodal.capabilities.clip_manifest import load_clips_from_index

extract_result = extract_clips(ctx, BAG_PATH, client=client, clip_config=clip_cfg)
clips = load_clips_from_index(ctx.clips_index_path)
assert clips, "extract 未产出 clip"
clip = clips[0]

print(json.dumps({
    "capability": "extract",
    "clip_rows": extract_result.clip_rows,
    "clips_index": str(extract_result.clips_index),
    "clip_id": clip.clip_id,
    "duration_sec": round(clip.duration_sec, 3),
    "audio": getattr(clip.audio, "audio_path", None) if clip.audio else None,
    "mel_png": clip.acoustic_panel_path,
    "mel_csv": clip.mel_matrix_path,
    "mp4": clip.clip_video_path,
}, ensure_ascii=False, indent=2))

### 2.1 预览：音频 / Mel / 视频

In [ ]:
from IPython.display import Audio, Image, Markdown, Video, display

if clip.audio and clip.audio.audio_path and Path(clip.audio.audio_path).is_file():
    display(Audio(filename=str(clip.audio.audio_path)))
if clip.acoustic_panel_path and Path(clip.acoustic_panel_path).is_file():
    display(Image(filename=clip.acoustic_panel_path))
if clip.mel_feature_text:
    display(Markdown("**mel_feature_text → MC label/embed text 侧**"))
    print(clip.mel_feature_text[:800])
if clip.clip_video_path and Path(clip.clip_video_path).is_file():
    display(Video(filename=clip.clip_video_path, embed=True, width=640))

## 3. sdk_asr — MC AI Function 转写

In [ ]:
from oms_multimodal import transcribe_clips

tr = transcribe_clips(ctx, client)
print("asr rows:", tr.row_count, "errors:", tr.errors)
print("asr.jsonl ->", ctx.asr_path)

clips = load_clips_from_index(ctx.clips_index_path)
clip = clips[0]
print("\nclip.asr_text:", (clip.asr_text or "")[:500])

## 4. sdk_label — MC AI Function 打标

输入含 ASR + events + **mel_feature_text** + taxonomy。

In [ ]:
from oms_multimodal import label_clips

lr = label_clips(ctx, client, run_asr=False, merge_asr_file=True)
print("label rows:", lr.row_count, "errors:", lr.errors)
print("labels.jsonl ->", ctx.labels_path)

label_row = json.loads(ctx.labels_path.read_text(encoding="utf-8").splitlines()[0])
print("\nscene_summary:", label_row.get("scene_summary", "")[:400])
print("mel_feature_included:", label_row.get("mel_feature_included"))
print("sample label keys:", list((label_row.get("labels") or {}).keys())[:8])

## 5. sdk_embed — MC AI Function 融合向量

In [ ]:
from oms_multimodal import embed_clips

er = embed_clips(ctx, client)
print("embedding rows:", er.row_count, "errors:", er.errors)
print("fusion_embeddings.jsonl ->", ctx.embeddings_path)

emb_row = json.loads(ctx.embeddings_path.read_text(encoding="utf-8").splitlines()[0])
vec = emb_row.get("embedding") or []
inputs = emb_row.get("inputs") or {}
print("model:", emb_row.get("model"), "dim:", emb_row.get("dimension"), "len:", len(vec))
print("mel_matrix:", inputs.get("mel_matrix_path"))
print("text preview:", (inputs.get("text") or "")[:400])

## 6. sdk_preview + 释放 session

In [ ]:
from oms_multimodal import materialize_preview

materialize_preview(ctx)
preview_dir = ctx.run_dir / "preview"
print("preview dir:", preview_dir)
if preview_dir.is_dir():
    for p in sorted(preview_dir.iterdir()):
        print(" ", p.name, p.stat().st_size if p.is_file() else "dir")

client.close()
print("MaxFrame session closed.")

## 7. （可选）sdk_infer 复合一步

等价 `infer_full` / `sdk_infer_node.py`；会重新跑整条 MC 链，默认跳过。

In [ ]:
if os.getenv("SDK_DEMO_RUN_INFER_FULL", "0") != "1":
    print("跳过 infer_full。设置 SDK_DEMO_RUN_INFER_FULL=1 启用。")
else:
    from oms_multimodal import infer_full

    infer_out = OUT_DIR / "infer_full_run"
    infer_out.mkdir(parents=True, exist_ok=True)
    c2, _, _ = build_sdk_client(backend="mc", require_taxonomy=True, work_dir=infer_out / "_sdk_work")
    c2.acoustic_panel_config = acoustic_cfg
    c2.clip_video_config = video_cfg
    c2.asr_config = asr_cfg
    ctx2 = make_run_context(c2, infer_out, clip_id=CLIP_ID, run_id=RUN_ID + "-infer", media_mode="local")
    try:
        full = infer_full(ctx2, BAG_PATH, c2, clip_config=clip_cfg)
        print(json.dumps({
            "label_rows": full.label_rows,
            "embedding_rows": full.embedding_rows,
            "errors": full.errors,
        }, ensure_ascii=False, indent=2))
    finally:
        c2.close()

---

### 常见问题

1. **`ODPS_*` 缺失** — 根 `.env` 或本目录 `.env`
2. **Python 3.14** — 用 3.11 venv + `pip install -e "../../piplinesdk[mc]"`
3. **上云 DPE** — DataWorks 另需 `DPE_IMAGE`、`OSS_RAM_ROLE_ARN`；本 notebook local extract 可不填
4. **与 API 无关** — 本演示不走 DashScope；HMI local worker 才用 `MODEL_BACKEND=api`